# NBA 3.1 HCP features: split outputs into cohort sets

Splits each HCP feature output of NB01-NB06 (Delta, one row per HCP x `COHORT_ID`) into Parquet sub files of consecutive cohorts. Decision (Saurabh, 2026-11-05): 20 cohorts for now, in 5 sub files of 4 cohorts each per table (`set_01_2025-01_to_2025-04` ... `set_05_2026-05_to_2026-08`). Settings and the reader `read_cohort_sets(table)` are in `00_cohort_config` (section HCP cohort sets).

* **Full run** (`VALIDATION_MODE = false`): reads the 3.1 HCP outputs (`hcp_claims_features/`), writes `hcp_claims_features/cohort_sets/4_cohorts_per_set/<table>/<set>/`. Run after NB06.
* **Validation run** (default): reads the latest full outputs (`COHORT_SET_SOURCE_DIR`, default the 2026-11-04 full run in the 3.0 volume, read only), keeps the 10 test HCPs, writes `hcp_claims_features/cohort_sets_validation/4_cohorts_per_set/`.
* The set size is part of the folder, so a split with another `COHORTS_PER_SET` never mixes with this one (`read_cohort_sets` cannot read a cohort twice). A rerun with the same size overwrites its sets.
* Job parameters: `COHORTS_PER_SET` (4), `EXPECTED_COHORTS` (20; empty = no check), `COHORT_SET_SOURCE_DIR`, `COHORT_SET_TABLES` (comma list, empty = all 11 tables).
* Validation, training and backtesting sets are not separated yet (open with Saurabh); every set is written the same way.

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"


In [0]:
%run "../../00_cohort_config"

In [0]:
# 1. Settings (job parameters, see the title cell) and the split function
import json, os
import pandas as pd

# Source of the HCP feature outputs. Full run: the 3.1 HCP outputs of this run. Validation run: the latest full
# outputs (2026-11-04 full run, still in the 3.0 volume: read only), filtered to the 10 test HCPs below.
SOURCE_DIR = get_run_param("COHORT_SET_SOURCE_DIR",
                           HCP_VOLUME + "notebook_output/" if VALIDATION_MODE else HCP_NOTEBOOK_OUTPUT)
SOURCE_DIR = SOURCE_DIR if SOURCE_DIR.endswith("/") else SOURCE_DIR + "/"
_tables = get_run_param("COHORT_SET_TABLES", None)
TABLES = [t.strip() for t in _tables.split(",")] if _tables else list(HCP_COHORT_TABLES)
assert set(TABLES) <= set(HCP_COHORT_TABLES), f"Unknown tables: {sorted(set(TABLES) - set(HCP_COHORT_TABLES))}"

# Validation run: the 10 test HCPs (NPI tables) and their PROVIDER_IDs (provider tables, same NPI mapping as NB01-NB06)
TEST_KEYS = None
if VALIDATION_MODE:
    _pids = [r[0] for r in spark.read.format("delta").load(NPI_BHID_MAPPING_PATH)
             .filter(F.col("NPI_NUMBER").cast("string").isin(get_test_npis()))
             .select(F.col("PROVIDER_ID").cast("string")).distinct().collect()]
    TEST_KEYS = {"NPI_NUMBER": get_test_npis(), "PROVIDER_ID": _pids}

print(f"VALIDATION_MODE  = {VALIDATION_MODE}" + (f" ({len(TEST_KEYS['NPI_NUMBER'])} NPIs, {len(TEST_KEYS['PROVIDER_ID'])} PROVIDER_IDs)" if TEST_KEYS else ""))
print(f"SOURCE_DIR       = {SOURCE_DIR}")
print(f"OUTPUT           = {HCP_COHORT_SET_OUTPUT}<table>/<set>/  (Parquet)")
print(f"COHORTS_PER_SET  = {COHORTS_PER_SET} | EXPECTED_COHORTS = {EXPECTED_COHORTS} | tables: {len(TABLES)}")


def read_source(table):
    """The HCP output of a table (all cohorts in the Delta log), filtered to the test HCPs in a validation run."""
    _path = SOURCE_DIR + table
    assert os.path.exists(_path + "/_delta_log"), f"{table}: no Delta table at {_path} (run NB01-NB06 first, or set COHORT_SET_SOURCE_DIR)"
    df = spark.read.format("delta").load(_path)
    key = HCP_COHORT_TABLES[table]
    assert {key, "COHORT_ID"} <= set(df.columns), f"{table}: columns {key} / COHORT_ID missing"
    return df if TEST_KEYS is None else df.filter(F.col(key).cast("string").isin(TEST_KEYS[key]))


def split_table(table):
    """Write one Parquet folder per cohort set; check rows written = rows of the source per set. Returns the manifest rows."""
    _path = SOURCE_DIR + table
    # Sets come from the cohorts of the whole table (Delta log, not the folders: replaceWhere keeps old
    # partition folders until VACUUM), so every table and the validation run get the same sets.
    all_ids = [r[0] for r in spark.read.format("delta").load(_path).select("COHORT_ID").distinct().collect()]
    if EXPECTED_COHORTS:
        assert len(all_ids) == EXPECTED_COHORTS, f"{table}: {len(all_ids)} cohorts, expected {EXPECTED_COHORTS}: {sorted(all_ids)}"
    sets = make_cohort_sets(all_ids)
    # The output folder holds the set size (<N>_cohorts_per_set/, 00_cohort_config), so another set size never
    # shares a folder. Set folders that do not belong to this split would be read twice by read_cohort_sets: stop.
    _base = HCP_COHORT_SET_OUTPUT + table + "/"
    _old = sorted(d for d in os.listdir(_base) if d.startswith("set_")) if os.path.exists(_base) else []
    _stale = sorted(set(_old) - {n for n, _ in sets})
    assert not _stale, f"{table}: {_base} has set folders that are not in this split {_stale}; check the cohorts"
    src = read_source(table)
    rows = []
    for name, ids in sets:
        out = assert_31_write_path(cohort_set_path(table, name), "cohort set")
        part = src.filter(F.col("COHORT_ID").isin(ids))
        part.write.mode("overwrite").parquet(out)
        n_src, n_out = part.count(), spark.read.parquet(out).count()
        assert n_src == n_out, f"{table} {name}: {n_src:,} source rows, {n_out:,} written"
        rows.append({"TABLE": table, "SET": name, "FIRST": ids[0], "LAST": ids[-1], "N_COHORTS": len(ids), "ROWS": n_out})
    with open(_base + "_manifest.json", "w") as fh:   # which cohorts are in which set, for the readers
        json.dump({"source": _path, "validation_mode": VALIDATION_MODE, "cohorts_per_set": COHORTS_PER_SET,
                   "sets": {n: ids for n, ids in sets}}, fh, indent=1)
    return rows

In [0]:
# 2. Split: one Parquet folder per table and cohort set (overwrite), plus <table>/_manifest.json
manifest = []
for t in TABLES:
    manifest += split_table(t)
    print(f"{t}: {', '.join(r['SET'] for r in manifest if r['TABLE'] == t)}")
manifest = pd.DataFrame(manifest)
display(manifest)

In [0]:
# 3. Independent check, read back with read_cohort_sets (00_cohort_config), per table:
#    rows and (key, COHORT_ID) pairs equal the source; every cohort is in exactly one set and in the set its name says;
#    sets are consecutive and have COHORTS_PER_SET cohorts (the last one may be smaller); same columns and types as the source.
checks = []
for t in TABLES:
    key = HCP_COHORT_TABLES[t]
    src, back = read_source(t), read_cohort_sets(t)
    n_src, n_back = src.count(), back.count()
    keys_diff = (src.select(F.col(key).cast("string"), "COHORT_ID").exceptAll(back.select(F.col(key).cast("string"), "COHORT_ID")).count()
                 + back.select(F.col(key).cast("string"), "COHORT_ID").exceptAll(src.select(F.col(key).cast("string"), "COHORT_ID")).count())
    per_set = {}
    for n in sorted(d for d in os.listdir(HCP_COHORT_SET_OUTPUT + t) if d.startswith("set_")):
        per_set[n] = sorted(r[0] for r in spark.read.parquet(cohort_set_path(t, n)).select("COHORT_ID").distinct().collect())
    flat = [c for ids in per_set.values() for c in ids]
    name_ok = all(n.endswith(f"{ids[0].split('_')[0]}_to_{ids[-1].split('_')[0]}") for n, ids in per_set.items() if ids)
    sizes = [len(json.load(open(HCP_COHORT_SET_OUTPUT + t + "/_manifest.json"))["sets"][n]) for n in per_set]
    checks.append({"TABLE": t, "SOURCE_ROWS": n_src, "SET_ROWS": n_back, "KEY_DIFF": keys_diff, "SETS": len(per_set),
                   "SET_SIZES": sizes, "COHORTS_IN_DATA": len(flat), "DUP_COHORTS": len(flat) - len(set(flat)),
                   "NAMES_OK": name_ok, "SAME_SCHEMA": dict(src.dtypes) == dict(back.dtypes)})
checks = pd.DataFrame(checks)
display(checks)
_bad = checks[(checks.SOURCE_ROWS != checks.SET_ROWS) | (checks.KEY_DIFF != 0) | (checks.DUP_COHORTS != 0)
              | ~checks.NAMES_OK | ~checks.SAME_SCHEMA
              | checks.SET_SIZES.map(lambda s: any(x != COHORTS_PER_SET for x in s[:-1]) or s[-1] > COHORTS_PER_SET)]
assert _bad.empty, f"Cohort-set check failed for: {_bad.TABLE.tolist()}"
print(f"All {len(checks)} tables OK: {checks.SETS.iloc[0]} sets of {COHORTS_PER_SET} cohorts, rows and keys equal the source")